In [1]:
import numpy as np
import pandas as pd
from scipy.special import logsumexp,gammaln
import random
from scipy.optimize import minimize
import tskit

n = 20  # Number of samples or lineages at start 
# N_e1 = 1500  # Effective population size in interval [0, T1]
# N_e2 = 500  # Effective population size in interval [T1, T2]
# N_e3 = 2500   # Effective population size in interval [T2, infinity]
T1 = 300    # First time boundary in generations
T2 = 350
a_values = np.arange(0, n + 1)
b_values = np.arange(0,n + 1)
def safe_logsumexp(log_terms,signs=None):
    if len(log_terms) == 0:
        return -np.inf 
    max_x = np.max(log_terms) 
    scale=max_x
    if max_x<np.log(1.0e-40):
        scale=np.log(1.0e-40) 
    shifted_exps = np.exp(log_terms - scale)# Compute e^(-(x_i - min_x))
    if signs is not None:
        shifted_exps *= signs  # Apply sign adjustments if given
    inner_sum = np.sum(shifted_exps)  # Sum up scaled exponentials
    if inner_sum <= 0:
        return -np.inf  # Prevent log(0) issues
    log_inner_sum = np.log(inner_sum) + scale  # Compute final log sum exp
    return log_inner_sum

# Efficiently calculate rising factorial in log space
def log_rising_factorial(a, b):
    if b == 0 or a==0:
        return 0
    if a < b:
        return 0  
    return np.sum([np.log(a - i) for i in range(b)])

# Efficiently calculate falling factorial in log space
def log_falling_factorial(a, n):
    if n == 0 or a==0:
        return 0
    return np.sum([np.log(a + i) for i in range(n)])

# Generate dataframes for rising and falling factorials
def create_factorial_dataframes(a_values, b_values):
    rising_data = {
        f'a[{b}]': [log_rising_factorial(a, b) for a in a_values] for b in b_values
    }
    falling_data = {
        f'a_{n}': [log_falling_factorial(a, n) for a in a_values] for n in b_values
    }
    df_rising = pd.DataFrame(rising_data, index=a_values)
    df_falling = pd.DataFrame(falling_data, index=a_values)
    return df_rising, df_falling

# Function to calculate "choose 2 out of k" (k choose 2)
def choose_2_out_of_k(k):
    if k < 2:
        return 0
    return k*(k-1)/2
C1 = np.array([choose_2_out_of_k(k) for k in range(n + 1)])


df_rising, df_falling = create_factorial_dataframes(a_values, b_values)  



def logconstant_tables_Anp(n,Ne=1,t=1):
    signs_arr = np.empty((n + 1, n + 1), dtype=object)
    log_factorial=np.empty((n + 1, n + 1), dtype=object)
    exp_arr=np.empty((n + 1, n + 1), dtype=object)
    logbk=np.empty((n+1), dtype=object)
    ck=np.empty(n )
    for k_start in range(1,n+1):
        logbk_temp=[]
        for k_end in range(1,k_start+1):
            log_terms= []
            signs=[]
            exp_term=[]
            ck[k_end-1]=-C1[k_end]*t/2
            logbk_temp.append(df_rising.iloc[k_start, k_end]-df_falling.iloc[k_start, k_end]+np.log(2*k_end-1))
            for k in range(k_end, k_start + 1):
                # Numerator and denominator in log space
                numerator = (
                    np.log(2 * k - 1)
                    + df_falling.iloc[k_end, k - 1]
                    + df_rising.iloc[k_start, k]
                )

                denominator = (
                gammaln(k_end + 1)  # log(k_end!)
                + gammaln(k - k_end + 1)  # log((k - k_end)!)
                + df_falling.iloc[k_start, k]
                )
                signs.append(1-2*((k - k_end)%2))
                log_terms.append(numerator - denominator)
                exp_term.append(-C1[k]*t/2/Ne)

            signs_arr[k_start, k_end] = np.array(signs)
            log_factorial[k_start, k_end] = np.array(log_terms)
            exp_arr[k_start, k_end] = np.array(exp_term)
            logbk[k_start]=np.array(logbk_temp)
    return signs_arr, log_factorial,exp_arr,logbk,ck

signs_arr, log_factorial,exp_arr,logbk,ck = logconstant_tables_Anp(n)

def Ant_p(Ne,k_start, k_end,t):
    log_terms=exp_arr[k_start,k_end]*t/Ne+log_factorial[k_start,k_end]
    signs=signs_arr[k_start,k_end]
    # Combine positive and negative parts safely
    log_inner_sum = safe_logsumexp(log_terms,signs=signs)
    return log_inner_sum



def log_Postk_in_interval(data,T1,T2,kth,n,Ne):  
    log_terms= []
    k_end=n-kth+1
    used_time=data[(data <= T2) & (data > T1)]
    k_start=n
    for i in range(len(used_time)):
        t=used_time[i]-T1
        log_terms.append(Ant_p(Ne,k_start,k_end,t))        
    log_inner_sum = safe_logsumexp(log_terms)+ np.log((n-kth+1)*(n-kth)/4/Ne)
    return log_inner_sum

def dynamic_programming(n, N_e1, N_e2, N_e3, T1, T2):
    N_e1 = float(N_e1)
    N_e2 = float(N_e2)
    N_e3 = float(N_e3)
    t = np.array([T1, T2 - T1])
    num_interval = len(t) + 1
    Ne = np.array([N_e1, N_e2, N_e3])
    max_sum = n - 1

    # Initialize log_dp table: all -inf, except base case
    log_dp = np.full((max_sum + 1, num_interval), -np.inf)
    log_dp[0][0] = 0.0  # log(1) = 0

    # Fill log_dp table
    for i in range(1, num_interval):
        for s in range(max_sum + 1):
            terms = []
            for j in range(s + 1):
                log_prev = log_dp[s - j][i - 1]
                if np.isneginf(log_prev):
                    continue
                ant_val = Ant_p(Ne[i - 1], n - (s - j), n - s, t[i - 1])
                log_term = log_prev + ant_val
                terms.append(log_term)
            if terms:
                log_dp[s][i] = safe_logsumexp(np.array(terms))

    return log_dp


In [2]:
def log_posexp_1(data,T1,n,N_e1):
    log_Post_expectation=[]
    for i in range(1,n):#i+1=1,2.....19
        p_temp=log_Postk_in_interval(data[i-1,:],0,T1,i,n,N_e1)
        log_Post_expectation.append(p_temp)
    return log_Post_expectation
def log_posexp_2(data,T1,T2,n,N_e2,dp):
    log_Post_expectation=[]
    for i in range(1,n):#i+1=1,2.....19
        log_P_total_i=[]
        for s in range(i):#s=0,1,2,....18
            prob_B_given_A = log_Postk_in_interval(data[i-1,:],T1,T2,i-s,n-s,N_e2)
            prob_A_s = dp[s][1]                                   #i+1 here should also minus s>
            log_P_total_i.append(prob_A_s+prob_B_given_A)
        log_Post_expectation.append(log_P_total_i)
    return log_Post_expectation
def log_posexp_3(data,T2,n,N_e3,dp,T3=np.inf):
    log_Post_expectation=[]
    for i in range(1,n):
        log_P_total_i=[]
        for s in range(i):
            prob_A_s = dp[s][2]
            prob_B_given_A = log_Postk_in_interval(data[i-1,:],T2,T3,i-s,n-s,N_e3)
            log_P_total_i.append(prob_A_s+prob_B_given_A)
        log_Post_expectation.append(log_P_total_i)
    return log_Post_expectation

def normalize_log_posterior(data,T1,T2,n,N_e1,N_e2,N_e3):
    dp=dynamic_programming(n,N_e1,N_e2,N_e3,T1,T2)
    log_posterior1,log_posterior2,log_posterior3=log_posexp_1(data,T1,n,N_e1),log_posexp_2(data,T1,T2,n,N_e2,dp),log_posexp_3(data,T2,n,N_e3,dp)
    # print(log_posterior)
    for i in range(len(log_posterior1)):
        constant=safe_logsumexp(np.array([log_posterior1[i]]+log_posterior2[i]+log_posterior3[i]))
        log_posterior1[i]=log_posterior1[i]-constant
        log_posterior2[i]=safe_logsumexp(log_posterior2[i])-constant
        log_posterior3[i]=safe_logsumexp(log_posterior3[i])-constant
    col1_pos=np.exp(log_posterior1)
    col2_pos=np.exp(log_posterior2)
    col3_pos=np.exp(log_posterior3)
    return col1_pos,col2_pos,col3_pos

In [3]:
#input data, run only once
file_path = "/space/s1/KaiyuanLi/Ne_estimate/ratio1_ran6/arg50_1.trees"
ts = tskit.load(file_path)
# Path to the .trees file
num_tree=ts.num_trees
num_arg=100
span=[]
# Load the tree sequence
time=np.zeros([n-1,ts.num_trees,num_arg])
n2count=0
for j in range(1,2):
    file_path = f"/space/s1/KaiyuanLi/Ne_estimate/ratio1_ran6/arg50_{j+1}.trees"
    ts = tskit.load(file_path)
    # Iterate over each tree and find the 19th coalescence time
    for tree in ts.trees():
        coalescence_times = [
        ts.node(node).time for node in tree.nodes()
        if  ts.node(node).time > 0]
        # Sort times in ascending order
        coalescence_times.sort()
        if j==1:
            span.append(int(tree.interval[1] - tree.interval[0]))
        # print(tree.index)
        # Get the 19th coalescence time (if enough events exist)
        if len(coalescence_times) == n-1:
            for i in range(n-1):
                time[i,tree.index,j]=coalescence_times[i]
                if T1 <= coalescence_times[i]<T2:
                    n2count+=1
                

# label the node and caluclate the node specific 
# some labeling of the arg.

In [4]:
# outdir='/space/s1/KaiyuanLi/Ne_estimate/ratio1/50'
# np.save(f"{outdir}/time.npy", time)
time=np.load('/space/s1/KaiyuanLi/Ne_estimate/ratio1_ran6/time.npy')

In [25]:
import numpy as np
from scipy.optimize import minimize
import random

# --- Precompute constants ---
def dp_to_p(dp, n):
    p_interval1 = np.zeros(n - 1)
    p_interval2 = np.zeros(n - 1)
    p_interval3 = np.zeros(n - 1)
    for i in range(1, n):
        for s in range(i, n):
            p_interval1[i - 1] += np.exp(dp[s][1])
        for s in range(i):
            p_interval3[i - 1] += np.exp(dp[s][2])
        p_interval2[i - 1] = max(0, 1 - min(p_interval1[i - 1], 1) - min(p_interval3[i - 1], 1))
    return np.log(p_interval1), np.log(p_interval2), np.log(p_interval3)

# --- Partial objective function (Ne3 fixed) ---
def joint_objective_Ne_partial(x, Ne3, T1, T2, subspan_rate, col1, col2, col3, n):
    Ne1, Ne2 = x
    dp_temp = dynamic_programming(n, Ne1, Ne2, Ne3, T1, T2)
    logp1, logp2, logp3 = dp_to_p(dp_temp, n)

    logp1 = np.where(np.isneginf(logp1), -1e30, logp1)
    logp2 = np.where(np.isneginf(logp2), -1e30, logp2)
    logp3 = np.where(np.isneginf(logp3), -1e30, logp3)

    total = 0
    for i in range(len(subspan_rate)):
        total += subspan_rate[i] * (
            np.sum(col1[i] * logp1) +
            np.sum(col2[i] * logp2) +
            np.sum(col3[i] * logp3)
        )
    return -total

# --- Ne3 estimator from posterior ---
def estimate_last_Ne(col3, time, span, T2):
    t_3 = []
    for tree in range(time.shape[1]):
        for rate in col3:
            for num_arg in range(time.shape[2]):
                adjusted_values = np.average(time[:, tree, num_arg], weights=rate) - T2
                if adjusted_values > 0:
                    t_3.extend([adjusted_values] * span[tree])
    return np.mean(t_3)

# --- EM setup ---
ran_num = 100
Ne1, Ne2, Ne3 = 2000, 2000, 2000
tolerance = 3
max_iter = 100
seed = 1
results = []
random.seed(seed)
selected_indices = random.sample(range(ts.num_trees), ran_num)
subspan = np.array(span)[selected_indices]
subspan_rate = subspan / np.sum(subspan)
print("Start EM")

for it in range(max_iter):

    col1, col2, col3 = [], [], []
    for i in selected_indices:
        c1, c2, c3 = normalize_log_posterior(time[:, i, :], T1, T2, n, Ne1, Ne2, Ne3)
        col1.append(np.array(c1))
        col2.append(np.array(c2))
        col3.append(np.array(c3))

    # Estimate Ne3 externally from col3
    

    # Minimize only over Ne1, Ne2
    res = minimize(
        lambda x: joint_objective_Ne_partial(x, Ne3_new, T1, T2, subspan_rate, col1, col2, col3, n),
        x0=np.array([Ne1, Ne2]),
        method='L-BFGS-B',
        bounds=[(300, 3000), (300, 3000)]
    )
    Ne1_new, Ne2_new = res.x
    loss_total = res.fun
    Ne3_new = estimate_last_Ne(col3, time[:, selected_indices, :], subspan, T2)
    print(f"[Iter {it + 1}] Ne1 = {Ne1_new:.2f}, Ne2 = {Ne2_new:.2f}, Ne3 = {Ne3_new:.2f}")
    print(f"Loss: {loss_total:.2f}")

    results.append((Ne1_new, Ne2_new, Ne3_new, loss_total))

    delta = abs(Ne1_new - Ne1) + abs(Ne2_new - Ne2) + abs(Ne3_new - Ne3)
    if delta < tolerance:
        break

    Ne1, Ne2, Ne3 = Ne1_new, Ne2_new, Ne3_new
    seed += 1


Start EM
[Iter 1] Ne1 = 1461.34, Ne2 = 1417.54, Ne3 = 2233.71
Loss: 4.48
[Iter 2] Ne1 = 1340.47, Ne2 = 1120.18, Ne3 = 2379.75
Loss: 4.61
[Iter 3] Ne1 = 1316.35, Ne2 = 946.70, Ne3 = 2442.37
Loss: 4.72
[Iter 4] Ne1 = 1318.63, Ne2 = 839.70, Ne3 = 2474.18
Loss: 4.79
[Iter 5] Ne1 = 1327.44, Ne2 = 769.27, Ne3 = 2493.36
Loss: 4.84
[Iter 6] Ne1 = 1336.21, Ne2 = 724.36, Ne3 = 2506.47
Loss: 4.88
[Iter 7] Ne1 = 1344.58, Ne2 = 692.38, Ne3 = 2515.35
Loss: 4.90
[Iter 8] Ne1 = 1344.58, Ne2 = 692.38, Ne3 = 2521.85
Loss: 4.92
[Iter 9] Ne1 = 1344.58, Ne2 = 692.38, Ne3 = 2522.14
Loss: 4.92
